# DICOM Tags Table Creation and Vector Search Database

This notebook aims to create the `dicom_tags` table, establish a vector search database for it, and provide comprehensive instructions to create a powerful Genie space. The steps include retrieving DICOM tags, extracting tag values, and ensuring accurate identification of patient and scan data.

In [0]:
%pip install databricks-vectorsearch==0.64 databricks-sdk==0.88
dbutils.library.restartPython()

In [0]:
%run ./config/proxy_prep

In [ ]:
sql_warehouse_id, table = init_widgets(show_volume=False)
init_env()

catalog, schema, table_name = table.split(".")
vs_endpoint = "pixels_vs_endpoint"

# Use the same catalog from job parameters for VS source tables
vs_catalog = catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {vs_catalog}.{schema}")

# Create **dicom_tags** table.
This table will contain all the dicom tags and description available in the standard

In [ ]:
import json
import os

_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

_dicom_tags_table = f"{vs_catalog}.{schema}.dicom_tags"

if not spark.catalog.tableExists(_dicom_tags_table):
    _ndjson_path = os.path.join(_repo_root, 'dbx', 'pixels', 'resources', 'dicom_tags.ndjson')
    with open(_ndjson_path, 'r') as f:
        tags = [json.loads(line) for line in f]

    tags_df = spark.createDataFrame(tags)

    # Write to main catalog to avoid Control Tower row filters that block VS delta sync indexes
    tags_df.select("Tag","Name","Keyword","VR","VM","Retired").write.saveAsTable(_dicom_tags_table)
    spark.sql(f"ALTER TABLE {_dicom_tags_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    print(f"Created {_dicom_tags_table}")
else:
    print(f"{_dicom_tags_table} already exists, skipping write")

# Table comment (idempotent)
spark.sql(f"""COMMENT ON TABLE {_dicom_tags_table} IS
  'DICOM Data Dictionary — every standard tag, keyword, value representation (VR), and value multiplicity (VM). '
  'Sourced from the DICOM PS3.6 standard. Used by the Pixels Genie space and Vector Search index for tag lookup.'""")

# Column comments (idempotent)
spark.sql(f"ALTER TABLE {_dicom_tags_table} ALTER COLUMN Tag COMMENT 'DICOM tag number in GGGGEEEE hex format (e.g. 00100010)'")
spark.sql(f"ALTER TABLE {_dicom_tags_table} ALTER COLUMN Name COMMENT 'Human-readable name of the DICOM attribute (e.g. Patient Name)'")
spark.sql(f"ALTER TABLE {_dicom_tags_table} ALTER COLUMN Keyword COMMENT 'CamelCase keyword used in DICOM protocol and code (e.g. PatientName)'")
spark.sql(f"ALTER TABLE {_dicom_tags_table} ALTER COLUMN VR COMMENT 'Value Representation — two-letter data-type code (e.g. PN, UI, LO)'")
spark.sql(f"ALTER TABLE {_dicom_tags_table} ALTER COLUMN VM COMMENT 'Value Multiplicity — allowed number of values (e.g. 1, 1-n, 3)'")
spark.sql(f"ALTER TABLE {_dicom_tags_table} ALTER COLUMN Retired COMMENT 'True if the tag is retired from the current DICOM standard'")

# Table tags (idempotent)
spark.sql(f"ALTER TABLE {_dicom_tags_table} SET TAGS ('phi' = 'false', 'accelerator' = 'pixels')")

# Create VectorSearch endpoint and vs index table
This step will guide you through creating a Vector Search endpoint, which enables efficient similarity search over DICOM tag embeddings. The endpoint will be used to power advanced search and retrieval capabilities in your Genie space.

In [ ]:
from databricks.vector_search.client import VectorSearchClient

# The following line automatically generates a PAT Token for authentication
client = VectorSearchClient()

try:
  client.get_endpoint(vs_endpoint)
except:
  client.create_endpoint(
      name=vs_endpoint,
      endpoint_type="STANDARD"
  )

# Create VS index (skip if it already exists from a previous run)
_vs_index_name = f"{vs_catalog}.{schema}.dicom_tags_vs"
try:
    index = client.get_index(vs_endpoint, _vs_index_name)
    print(f"VS index {_vs_index_name} already exists, skipping creation")
except:
    index = client.create_delta_sync_index(
      endpoint_name=vs_endpoint,
      source_table_name=f"{vs_catalog}.{schema}.dicom_tags",
      index_name=_vs_index_name,
      pipeline_type="TRIGGERED",
      primary_key="Tag",
      embedding_source_column="Name",
      embedding_model_endpoint_name="databricks-bge-large-en"
    )
    print(f"VS index {_vs_index_name} created")

# Create VS Functions

In [ ]:
import os
import os.path

_nb_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
_nb_dir = "/Workspace" + os.path.dirname(_nb_ctx.notebookPath().get())
_repo_root = os.path.normpath(os.path.join(_nb_dir, ".."))

sql_functions_path = os.path.join(_repo_root, "ai-bi", "genie", "CREATE_VS_FUNCTION.sql")

with open(sql_functions_path, "r") as f:
  sql_command = f.read()\
    .replace("{UC_SCHEMA}", f"{catalog}.{schema}")\
    .replace("{UC_DICOM_TAGS_VS}", f"{vs_catalog}.{schema}.dicom_tags_vs")
  spark.sql(sql_command)
  print("SQL command executed")

# Create GENIE SPACE

In [ ]:
import os
import requests
import json as _json

genie_api_endpoint = f"{os.environ['DATABRICKS_HOST']}/api/2.0/genie/spaces"
token = os.environ['DATABRICKS_TOKEN']

description = "A Genie space for exploring and querying DICOM pixel metadata, enabling natural language interaction and insights for medical imaging data."
title = "Pixels - Genie"

# Use the user's home directory in workspace for the Genie space parent path
_user = spark.sql("SELECT current_user()").collect()[0][0]
parent_path = f"/Workspace/Users/{_user}/genie/"

# Create the directory in workspace filesystem
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
try:
    w.workspace.mkdirs(parent_path)
except:
    pass

# Delete existing Genie spaces with the same title to avoid duplicates
_list_resp = requests.get(genie_api_endpoint, headers={"Authorization": f"Bearer {token}"})
if _list_resp.status_code == 200:
    for _space in _list_resp.json().get("spaces", []):
        if _space.get("title", "").startswith(title):
            _sid = _space["space_id"]
            _del_resp = requests.delete(f"{genie_api_endpoint}/{_sid}", headers={"Authorization": f"Bearer {token}"})
            print(f"Deleted existing Genie space: {_space['title']} (id={_sid}) — HTTP {_del_resp.status_code}")

serialized_space_file = os.path.join(_repo_root, "ai-bi", "genie", "serialized_space.json")

with open(serialized_space_file, "r") as f:
  # Replace UC_SCHEMA references, then fix VS tables that live in main catalog
  raw = f.read()\
    .replace("{UC_SCHEMA}", f"{catalog}.{schema}")\
    .replace(f"{catalog}.{schema}.dicom_tags_vs", f"{vs_catalog}.{schema}.dicom_tags_vs")\
    .replace(f"{catalog}.{schema}.dicom_tags", f"{vs_catalog}.{schema}.dicom_tags")
  
  # Re-sort tables by identifier (Genie API requires sorted order)
  space_obj = _json.loads(raw)
  if "data_sources" in space_obj and "tables" in space_obj["data_sources"]:
      space_obj["data_sources"]["tables"].sort(key=lambda t: t.get("identifier", ""))
  serialized_space = _json.dumps(space_obj)

  body = {
    "title": title,
    "description": description,
    "serialized_space": serialized_space,
    "warehouse_id": sql_warehouse_id,
    "parent_path": parent_path
  }

  resp = requests.post(genie_api_endpoint, headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"}, json=body)
  if resp.status_code != 200:
    raise Exception(f"Error creating Genie space: {resp.text}")
  else:
    resp_body = resp.json()
    print(f"Genie space created, id: {resp_body['space_id']} title: {resp_body['title']}")